### Install requiremnts

In [1]:
!pip install -r requirements.txt


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
from checkpoints import DT, MT, LT
import os
import requests
import zipfile
from validate import load_model, get_test_loader, test
from checkpoints import DT, MT, LT
import numpy as np

DOWN_TASK = "SNE"
REMOTE_MODEL_URL = "https://next.hessenbox.de/index.php/s/fYKk7FDG446jgxD/download"

/home/bellafkir/Documents/sa4birds/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#### Download a trained model


In [3]:
extract_dir = "checkpoints/DT"
zip_filename = "file.zip"
if not os.path.exists(f'{extract_dir}/{DOWN_TASK}'):

        # -----------------------
        # Download ZIP file
        # -----------------------
        with requests.get(REMOTE_MODEL_URL, stream=True) as r:
            r.raise_for_status()
            with open(zip_filename, "wb") as f:
                for chunk in r.iter_content(chunk_size=8192):
                    if chunk:
                        f.write(chunk)

        print("Download complete.")

        # -----------------------
        # Extract ZIP file
        # -----------------------
        os.makedirs(extract_dir, exist_ok=True)

        with zipfile.ZipFile(zip_filename, "r") as zip_ref:
            zip_ref.extractall(extract_dir)
        print(f"Unzipped to '{extract_dir}'")

In [4]:

for down_task in ["HSN", "POW", "SNE", "PER", "NES", "UHH", "NBP", "SSW"]:
    ckpts = DT.get(down_task, [])

    models = []
    configs = []

    # Load configs & models
    for ckpt in ckpts:
        m, cfg = load_model(ckpt)
        models.append(m)
        cfg.frontend.val_target_length = 701
        cfg.event_decoder.val.extracted_interval = 7
        cfg.train.dataset_name = down_task
        configs.append(cfg)

    results = dict()
    val_loader, class_names = get_test_loader(configs[0])

    label2id = {k: v for k, v in configs[0].train.label_map.items()}
    relevant = [label2id[c] for c in class_names]

    metrics = {"auroc": [],
               "cmap": [],
               "top1_acc": []}

    # test models
    for model, cfg in zip(models, configs):
        auroc, cmap, top1_acc = test(model, val_loader, cfg, relevant)
        metrics["auroc"].append(auroc)
        metrics["cmap"].append(cmap)
        metrics["top1_acc"].append(top1_acc)

    # Aggregate
    results[down_task] = {key: float(np.mean(values)) for key, values in metrics.items()}
    print(results)

number of test events in HSN: 12000


Testing: 100%|██████████| 94/94 [00:45<00:00,  2.08it/s]


{'HSN': {'auroc': 0.9582019252201605, 'cmap': 0.5805238184761927, 'top1_acc': 0.7330605586369833}}
number of test events in POW: 4560


Testing: 100%|██████████| 36/36 [00:23<00:00,  1.55it/s]


{'POW': {'auroc': 0.9422970928184865, 'cmap': 0.6007718514368877, 'top1_acc': 0.9446458220481873}}
number of test events in SNE: 23756


...

{'UHH': {'auroc': 0.883188941937096, 'cmap': 0.344212708470556, 'top1_acc': 0.6776004433631897}}
number of test events in NBP: 539


Testing: 100%|██████████| 5/5 [00:04<00:00,  1.15it/s]


{'NBP': {'auroc': 0.9274592879180369, 'cmap': 0.7067440309580988, 'top1_acc': 0.7056277195612589}}
number of test events in SSW: 205200


Testing:  94%|█████████▍| 1509/1604 [12:26<00:40,  2.32it/s]Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7fd8735f6f20>
Traceback (most recent call last):
  File "/home/bellafkir/Documents/sa4birds/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py", line 1479, in __del__
    self._shutdown_workers()
  File "/home/bellafkir/Documents/sa4birds/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py", line 1443, in _shutdown_workers
    w.join(timeout=_utils.MP_STATUS_CHECK_INTERVAL)
  File "/usr/lib/python3.12/multiprocessing/process.py", line 149, in join
    res = self._popen.wait(timeout)
          ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/popen_fork.py", line 40, in wait
    if not wait([self.sentinel], timeout):
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/connection.py", line 1136, in wait
    ready = selector.select(timeout)
            ^^^^^^^^^^^^^^^^^^^^^

KeyboardInterrupt: 

In [33]:

for down_task in ["HSN", "POW", "SNE", "PER", "NES", "UHH", "NBP", "SSW"]:
    ckpts  = MT['ALL']

    models = []
    configs = []

    # Load configs & models
    for ckpt in ckpts:
        m, cfg = load_model(ckpt)
        models.append(m)
        cfg.frontend.val_target_length = 701
        cfg.event_decoder.val.extracted_interval = 7
        cfg.train.dataset_name = down_task
        configs.append(cfg)

    results = dict()
    val_loader, class_names = get_test_loader(configs[0])

    label2id = {k: v for k, v in configs[0].train.label_map.items()}
    relevant = [label2id[c] for c in class_names]

    metrics = {"auroc": [],
               "cmap": [],
               "top1_acc": []}

    # test models
    for model, cfg in zip(models, configs):
        auroc, cmap, top1_acc = test(model, val_loader, cfg, relevant)
        metrics["auroc"].append(auroc)
        metrics["cmap"].append(cmap)
        metrics["top1_acc"].append(top1_acc)

    # Aggregate
    results[down_task] = {key: float(np.mean(values)) for key, values in metrics.items()}
    print(results)

number of test events in HSN: 12000


Testing: 100%|██████████| 94/94 [00:46<00:00,  2.03it/s]


{'HSN': {'auroc': 0.9184371456266923, 'cmap': 0.5579475516695079, 'top1_acc': 0.7075196901957194}}
number of test events in POW: 4560


Testing: 100%|██████████| 36/36 [00:24<00:00,  1.48it/s]


{'POW': {'auroc': 0.9375645350612535, 'cmap': 0.5368986922681116, 'top1_acc': 0.9144389033317566}}
number of test events in SNE: 23756


...

{'UHH': {'auroc': 0.885294418690198, 'cmap': 0.3412588201287024, 'top1_acc': 0.7122304836908976}}
number of test events in NBP: 539


Testing: 100%|██████████| 5/5 [00:04<00:00,  1.01it/s]


{'NBP': {'auroc': 0.935165767197601, 'cmap': 0.6946172203238211, 'top1_acc': 0.7167594234148661}}
number of test events in SSW: 205200


Testing: 100%|██████████| 1604/1604 [11:41<00:00,  2.29it/s]


{'SSW': {'auroc': 0.9728458534436427, 'cmap': 0.48455712328995015, 'top1_acc': 0.7581257025400797}}


In [34]:

for down_task in ["HSN", "POW", "SNE", "PER", "NES", "UHH", "NBP", "SSW"]:
    ckpts  = LT['ALL']

    models = []
    configs = []

    # Load configs & models
    for ckpt in ckpts:
        m, cfg = load_model(ckpt)
        models.append(m)
        cfg.frontend.val_target_length = 701
        cfg.event_decoder.val.extracted_interval = 7
        cfg.train.dataset_name = down_task
        configs.append(cfg)

    results = dict()
    val_loader, class_names = get_test_loader(configs[0])

    label2id = {k: v for k, v in configs[0].train.label_map.items()}
    relevant = [label2id[c] for c in class_names]

    metrics = {"auroc": [],
               "cmap": [],
               "top1_acc": []}

    # test models
    for model, cfg in zip(models, configs):
        auroc, cmap, top1_acc = test(model, val_loader, cfg, relevant)
        metrics["auroc"].append(auroc)
        metrics["cmap"].append(cmap)
        metrics["top1_acc"].append(top1_acc)

    # Aggregate
    results[down_task] = {key: float(np.mean(values)) for key, values in metrics.items()}
    print(results)

number of test events in HSN: 12000


Testing: 100%|██████████| 94/94 [00:57<00:00,  1.63it/s]


{'HSN': {'auroc': 0.9362847044702539, 'cmap': 0.5451335159817338, 'top1_acc': 0.7223978638648987}}
number of test events in POW: 4560


Testing: 100%|██████████| 36/36 [00:29<00:00,  1.23it/s]


{'POW': {'auroc': 0.9054063059801476, 'cmap': 0.5424785885572563, 'top1_acc': 0.9435130556424459}}
number of test events in SNE: 23756


...

{'UHH': {'auroc': 0.8974533207180894, 'cmap': 0.3206880047688036, 'top1_acc': 0.6917312343915304}}
number of test events in NBP: 539


Testing: 100%|██████████| 5/5 [00:05<00:00,  1.09s/it]


{'NBP': {'auroc': 0.9368651411143865, 'cmap': 0.7066477922374909, 'top1_acc': 0.7322201530138651}}
number of test events in SSW: 205200


Testing: 100%|██████████| 1604/1604 [14:11<00:00,  1.88it/s]


{'SSW': {'auroc': 0.9696315710574622, 'cmap': 0.45494037936116544, 'top1_acc': 0.7615264256795248}}
